In [0]:
import os, sys
sys.path.insert(0, os.path.join(os.path.dirname(os.getcwd()), "src"))
from databricks.sdk import WorkspaceClient
from lh import common, bi, publish

# t00_job_smoke: last task of the daily job. Works in any environment; no dev-only counts.
# FAIL stops the job (and marks the run failed); WARN is reported but does not fail.
dbutils.widgets.dropdown("env", "dev", ["dev", "test", "prod"])
env = dbutils.widgets.get("env")
cfg = common.load_config(env)
cat, code = cfg["catalog"], cfg["code"]
root = os.path.dirname(os.getcwd())
fact = bi.FACTS[code][0]
results = []


def check(level, name, ok, detail=""):
    status = "PASS" if ok else level
    results.append(status)
    print(f"{status:4} {name}", detail)


# 1. every stage of this run wrote a success in the last 6 hours, and nothing in it failed
#    (a stage is its own job name or its sub-jobs, e.g. bronze.patients; 'rejected' = a bad file set aside = WARN)
stages = ["bronze", "silver", "gold"] + (["group"] if code == "bhg" else []) + ["bi", "bi_publish"]
recent = spark.sql(f"""SELECT job, lower(last_status) AS s FROM {cat}.gold.v_health_runs
                       WHERE last_run >= current_timestamp() - INTERVAL 6 HOURS""").collect()
for stage in stages:
    rows = [r for r in recent if r.job == stage or r.job.startswith(stage + ".")]
    states = sorted({r.s for r in rows})
    bad = sorted({r.job for r in rows if r.s not in ("success", "rejected")})
    check("FAIL", f"stage {stage}: success in the last 6 hours, nothing failed", "success" in states and not bad, bad or states or "(no run)")
    if "rejected" in states:
        check("WARN", f"stage {stage}: no file set aside", False, sorted(r.job for r in rows if r.s == "rejected"))
rebuilt = [r for r in recent if r.job == "silver_rebuild"]
check("WARN", "Silver ran normally (no rebuild from Bronze needed)", not rebuilt,
      f"(rebuild {rebuilt[0].s}: the Silver checkpoint failed in this run, see the silver task)" if rebuilt else "")

# 2. Silver and Gold tables are not empty (except tables that are empty when all is well)
MAY_BE_EMPTY = ("quarantine", "group_match_review")   # nothing held back, no look-alike to review
tables = spark.sql(f"""SELECT table_schema, table_name FROM {cat}.information_schema.tables
                       WHERE table_schema IN ('silver', 'gold') AND table_type <> 'VIEW'
                         AND table_name NOT IN {MAY_BE_EMPTY}""").collect()
empty = [f"{t.table_schema}.{t.table_name}" for t in tables if spark.table(f"{cat}.{t.table_schema}.{t.table_name}").limit(1).count() == 0]
check("FAIL", f"{len(tables)} Silver/Gold tables have rows", not empty, empty or "")

# 3. the fact: unique keys, no event dated after tomorrow
f = spark.sql(f"""SELECT count(*) AS n, count(DISTINCT event_key) AS k,
                         sum(CASE WHEN date_key > CAST(date_format(date_add(current_date(), 1), 'yyyyMMdd') AS INT) THEN 1 ELSE 0 END) AS future
                  FROM {cat}.gold.{fact}""").first()
check("FAIL", f"{fact}: {f.n} rows, event keys unique", f.n > 0 and f.n == f.k, f"({f.n - f.k} duplicates)" if f.n != f.k else "")
check("FAIL", f"{fact}: no event dated after tomorrow", f.future == 0, f"({f.future})" if f.future else "")

# 4. every BI view of this catalog answers
broken = []
for view in bi.wanted(cfg):
    try:
        spark.table(f"{cat}.gold.{view}").limit(1).collect()
    except Exception as e:
        broken.append(f"{view}: {str(e)[:80]}")
check("FAIL", f"{len(bi.wanted(cfg))} BI views answer", not broken, broken or "")

# 5. the dashboards are published for this environment
api = publish.Api(WorkspaceClient())
bi_cfg = publish.load_bi(root)
folder = api.home() + bi_cfg["folder"][1:] if bi_cfg["folder"].startswith("~/") else bi_cfg["folder"]
for name, _ in publish.dashboards(root, env):
    did = api.find(f"{folder}/{env}/{name}_{env}.lvdash.json")
    check("FAIL", f"dashboard {name}_{env} published", bool(did) and bool(api.published(did).get("revision_create_time")))

# 6. freshness: dev may lag (runs on demand); test and prod must be at most 2 days behind
fresh = spark.sql(f"SELECT days_behind FROM {cat}.gold.v_health_freshness").first()
check("WARN" if env == "dev" else "FAIL", f"{fact} at most 2 days behind", fresh.days_behind <= 2, f"({fresh.days_behind} days)")

# 7. BHG only: the group merge is complete (behind = the child is late; reported, never blocks the parent)
if code == "bhg":
    m = spark.sql(f"SELECT status, backlog_days FROM {cat}.gold.v_health_merge ORDER BY run_ts DESC LIMIT 1").first()
    check("WARN", "group merge complete", m is not None and m.status == "complete", "" if m is None else f"({m.status}, backlog {m.backlog_days} days)")

fails, warns = results.count("FAIL"), results.count("WARN")
print(f"\n{results.count('PASS')}/{len(results)} PASS, {warns} WARN, {fails} FAIL")
common.log_run(spark, cfg, "smoke", "success" if not fails else "failed", len(results), f"pass {results.count('PASS')}, warn {warns}, fail {fails}")
assert not fails, "t00_job_smoke failed"